In [ ]:
import re
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD BIRKBECK SPELLING ERROR CORPUS
# ============================================================

# Path of the Birkbeck corpus file
# Change this path according to where you saved the corpus.
CORPUS_FILE = "spell-errors.txt"

try:
    with open(CORPUS_FILE, "r", encoding="latin-1") as file:
        corpus_text = file.read()

    print("Birkbeck corpus loaded successfully!")

except FileNotFoundError:
    print("Corpus file not found.")
    print("Please place 'spell-errors.txt' in the same folder as this Python file.")
    corpus_text = ""


# ============================================================
# 2. BUILD VOCABULARY OF CORRECTLY SPELLED WORDS
# ============================================================

vocabulary = set()

if corpus_text:

    lines = corpus_text.splitlines()

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # Birkbeck corpus commonly contains lines such as:
        # $correct_word
        # misspelled_word

        if line.startswith("$"):
            correct_word = line[1:].strip().lower()

            # Keep alphabetic words
            if correct_word.isalpha():
                vocabulary.add(correct_word)

print("Vocabulary size:", len(vocabulary))


# ============================================================
# 3. ADD COMMON NLP WORDS
# ============================================================

# These words help the demonstration work even if the corpus
# does not contain some words.

extra_words = {
    "machine",
    "learning",
    "course",
    "natural",
    "language",
    "processing",
    "artificial",
    "intelligence",
    "computer",
    "science",
    "student",
    "students",
    "data",
    "analysis",
    "model",
    "models",
    "deep",
    "neural",
    "network",
    "networks",
    "python",
    "algorithm",
    "algorithms",
    "classification",
    "sentiment",
    "text",
    "search",
    "engine"
}

vocabulary.update(extra_words)

print("Final vocabulary size:", len(vocabulary))


# ============================================================
# 4. LEVENSHTEIN EDIT DISTANCE
# ============================================================

def edit_distance(word1, word2):

    word1 = word1.lower()
    word2 = word2.lower()

    rows = len(word1) + 1
    cols = len(word2) + 1

    matrix = np.zeros((rows, cols), dtype=int)

    # Initialize first column
    for i in range(rows):
        matrix[i][0] = i

    # Initialize first row
    for j in range(cols):
        matrix[0][j] = j

    # Calculate edit distance
    for i in range(1, rows):

        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,       # Deletion
                matrix[i][j - 1] + 1,       # Insertion
                matrix[i - 1][j - 1] + cost # Substitution
            )

    return matrix[-1][-1]


# ============================================================
# 5. FIND CLOSEST MATCH
# ============================================================

def find_closest_word(word):

    word = word.lower()

    if not vocabulary:
        return word

    best_word = word
    best_distance = float("inf")

    for candidate in vocabulary:

        # Ignore candidates that are very different in length
        if abs(len(candidate) - len(word)) > 3:
            continue

        distance = edit_distance(word, candidate)

        if distance < best_distance:
            best_distance = distance
            best_word = candidate

    return best_word


# ============================================================
# 6. TOKENIZE SEARCH QUERY
# ============================================================

def tokenize_query(query):

    return re.findall(r"[a-zA-Z]+", query.lower())


# ============================================================
# 7. SPELLING CORRECTION
# ============================================================

def correct_query(query):

    tokens = tokenize_query(query)

    corrected_tokens = []

    corrections = []

    for word in tokens:

        if word in vocabulary:

            corrected_tokens.append(word)

        else:

            suggested_word = find_closest_word(word)

            distance = edit_distance(word, suggested_word)

            # Only correct when the candidate is reasonably close
            if distance <= 3:

                corrected_tokens.append(suggested_word)

                corrections.append(
                    {
                        "Incorrect Word": word,
                        "Suggested Correction": suggested_word,
                        "Edit Distance": distance
                    }
                )

            else:

                corrected_tokens.append(word)

    corrected_query = " ".join(corrected_tokens)

    return corrected_query, corrections


# ============================================================
# 8. DISPLAY RESULTS
# ============================================================

def display_result(query):

    corrected_query, corrections = correct_query(query)

    print("\n" + "=" * 60)
    print("SPELLING CORRECTOR")
    print("=" * 60)

    print("\nOriginal Query:")
    print(query)

    if corrections:

        print("\nIncorrect Words and Suggestions:")

        dataframe = pd.DataFrame(corrections)

        print(dataframe.to_string(index=False))

    else:

        print("\nNo spelling errors detected.")

    print("\nCorrected Query:")
    print(corrected_query)

    print("=" * 60)


# ============================================================
# 9. TEST WITH MULTIPLE SEARCH QUERIES
# ============================================================

test_queries = [
    "machne lerning cours",
    "artifical inteligence",
    "naturall languge procesing",
    "computr scince",
    "sentment anlysis"
]

print("\n\nTESTING MULTIPLE QUERIES")

for query in test_queries:
    display_result(query)


# ============================================================
# 10. ACCEPT USER INPUT
# ============================================================

print("\n\nUSER SEARCH QUERY")

user_query = input("Enter your search query: ")

display_result(user_query)